# Math 638 Project 2 - Homework 6.1

## Attention for Tokenized Phase-Space Dynamics

This report presents Homework 6.1 in the same formal deliverables style used for the previous Math 638 reports.

The project continues the Lorenz tokenization experiment from Chapter 5 and replaces fixed-memory Markov prediction with a learned causal-attention mechanism.

The main question is

$$
\boxed{
\text{Can causal attention use the past trajectory more flexibly than a fixed-order Markov model?}
}
$$

A second and equally important question is whether better one-step prediction produces better autonomous dynamics.

The complete workflow is

$$
\text{Lorenz trajectories}
\longrightarrow
\text{phase-space tokens}
\longrightarrow
\begin{cases}
\text{Markov baseline},\\
\text{causal attention model}
\end{cases}
\longrightarrow
\text{prediction and generation diagnostics}.
$$

All headline comparisons in this report use the corrected like-for-like evaluation protocol:

- all attention context lengths are compared on target positions 32 and later;
- the selected Markov and attention models are compared on those same positions;
- Markov and attention generators receive the same first 32 true held-out tokens;
- future held-out tokens are never supplied during autonomous generation.

## Executive Summary

The selected Lorenz system is deterministic, dissipative, and chaotic. For

$$
\sigma=10,\qquad \rho=28,\qquad \beta=\frac{8}{3},
$$

the divergence is

$$
\nabla\cdot F
=
-\sigma-1-\beta
=
-\frac{41}{3}<0.
$$

The experiment uses 40 training, 10 validation, and 10 test trajectories, each with 1,251 retained samples after discarding a transient. The baseline phase-space discretization is

$$
12\times12\times12,
$$

giving 1,728 possible tokens. In the saved run, 252 tokens occur in training and 262 occur across all splits.

The selected Markov baseline is $$m=3$$. On the corrected common target set beginning at position 32, its final test accuracy is

$$
0.758737.
$$

The validation-selected attention model uses context length

$$
L=32
$$

and 187,920 trainable parameters. On the same test target positions its accuracy is

$$
0.839212,
$$

an improvement of

$$
0.080476
$$

or about **8.05 percentage points**.

Attention is not merely concentrated on the immediately previous token. Across the held-out test set, the mean expected attention lag is approximately 13.25 steps for ordinary targets and 16.26 steps for lobe-switch targets. The switch-target mean is larger in every one of the ten test books.

However, better one-step prediction does **not** imply better autonomous Lorenz dynamics. Under the corrected equal-prefix generation experiment:

- true test trajectories switch lobes about 10.96 times per 1000 steps;
- Markov-generated trajectories switch about 10.76 times per 1000 steps;
- attention-generated trajectories switch about 25.22 times per 1000 steps.

Thus the attention model is substantially better for one-step prediction but substantially worse than the Markov generator on long-run lobe-switching behavior.

The strongest defensible conclusion is:

> **Causal attention learns useful nonlocal predictive correlations in the tokenized Lorenz trajectories, but higher next-token accuracy does not by itself recover the long-run Lorenz dynamics.**

## Deliverables Overview

| Section | Content | Status |
| --- | --- | --- |
| 1 | Lorenz choice, mathematical classification, trajectory generation, and phase-space discretization | Complete |
| 2 | Token sequence construction | Complete |
| 3 | Markov / m-gram baseline | Complete |
| 4 | Small causal attention predictor | Complete |
| 5 | Attention versus baseline comparison | Complete with common-target correction |
| 6 | Attention-weight visualization and temporal-lag analysis | Complete |
| 7 | Autonomous generation and Lorenz diagnostics | Complete with equal-prefix correction |
| 8 | Kirchhoff invariant test | Not applicable to Lorenz |
| 9 | Attention context-length comparison | Complete with common-target correction |
| 10 | LLM interpretation and human critique | Complete |
| 11 | Reproducibility and interpretation limits | Complete |

# 1. Lorenz System, Trajectories, and Phase-Space Discretization

The chosen dynamical system is

$$
\dot{x}=\sigma(y-x),
$$

$$
\dot{y}=x(\rho-z)-y,
$$

$$
\dot{z}=xy-\beta z,
$$

with

$$
\sigma=10,\qquad
\rho=28,\qquad
\beta=\frac{8}{3}.
$$

The divergence is

$$
\nabla\cdot F
=
-\sigma-1-\beta
=
-\frac{41}{3}<0,
$$

so the system is dissipative. For the classical parameter regime used here, the system is also chaotic.

The experiment generates independent trajectory books from seeded initial conditions:

| Split | Books | Samples per book | Total retained samples |
| --- | ---: | ---: | ---: |
| Training | 40 | 1,251 | 50,040 |
| Validation | 10 | 1,251 | 12,510 |
| Test | 10 | 1,251 | 12,510 |

A 5-time-unit transient is discarded and 25 time units are retained with

$$
\Delta t=0.02.
$$

The phase-space bounds are fitted from **training states only** and padded by 5%. The frozen bounds are approximately

$$
x\in[-20.9512,21.0698],
$$

$$
y\in[-28.7294,29.0481],
$$

$$
z\in[-0.0252,49.3156].
$$

Validation and test trajectories do not determine the grid.

![Example Lorenz trajectory](../result%20images/hw6_1/step1_example_lorenz_3d.png)


# 2. Token Sequence Construction

The baseline discretization uses

$$
12\times12\times12
$$

uniform phase-space boxes, so

$$
|\mathcal V_{\mathrm{possible}}|
=
12^3
=
1728.
$$

Each sampled continuous state

$$
X(t_i)
$$

is mapped to the box that contains it, producing

$$
X(t_1),X(t_2),\ldots,X(t_N)
\longrightarrow
a_1,a_2,\ldots,a_N.
$$

Every trajectory remains a separate token book. No Markov transition or attention window crosses a book boundary.

The neural softmax uses all 1,728 possible grid boxes as candidate tokens. This is mathematically valid even though only a subset is observed in the training data.

Saved-run vocabulary statistics:

- possible tokens: 1,728;
- observed in training: 252;
- observed across all splits: 262;
- clipped held-out samples: 0.

![Thirty most frequent phase-space tokens](../result%20images/hw6_1/step2_top30_token_frequencies.png)

# 3. Markov / m-gram Baseline


For order $$m$$,

$$
P(
a_{n+1}=b
\mid
a_{n-m+1},\ldots,a_n
)
=
\frac{
\#\{(a_{n-m+1},\ldots,a_n,b)\}
}{
\#\{(a_{n-m+1},\ldots,a_n)\}
}.
$$

Orders

$$
m=1,2,3
$$

are fitted from training books only. Validation selects

$$
m=3.
$$

For a strict comparison with the attention models, the final audit evaluates every Markov order on the same held-out target positions beginning at position 32.


| Markov order m | Common test positions | Common-support test accuracy |
| --- | --- | --- |
| 1 | 12190 | 0.616407 |
| 2 | 12190 | 0.699262 |
| 3 | 12190 | 0.758737 |

The selected baseline therefore has corrected common-support test accuracy

$$
0.758737.
$$

The common-support value is used in all final comparisons.

![Markov accuracy versus context](../result%20images/hw6_1/step3_markov_accuracy_vs_context.png)

The figure above was generated from the native per-order evaluation ranges. The corrected common-support table is the authoritative like-for-like comparison.


# 4. Small Causal Attention Predictor

The attention model contains exactly the main components requested by Homework 6.1:

- token embeddings;
- learned positional embeddings;
- one single-head causal self-attention layer;
- a small feed-forward residual block;
- a linear output layer whose cross-entropy training is equivalent to softmax next-token prediction.

For a context of length $$L$$,

$$
h_j
=
E(a_j)+p_j.
$$

Queries, keys, and values are

$$
q_j=W_Qh_j,\qquad
k_j=W_Kh_j,\qquad
v_j=W_Vh_j.
$$

The causal attention scores are

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt d},
\qquad
j\le i,
$$

and

$$
A_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{r\le i}\exp(s_{ir})}.
$$

The target token is excluded from the input window. The lower-triangular causal mask prevents every internal query from attending to future positions.

Training minimizes

$$
L(\theta)
=
-\frac{1}{M}
\sum_i
\log
P_\theta(
a_{i+1}
\mid
a_{i-L+1},\ldots,a_i
).
$$

Gradient updates use training books only. Validation is used for early stopping and context selection. The test books are loaded by the attention-training notebook only after model selection is fixed.

# 5. Attention Versus the Markov Baseline


The attention family tests

$$
L\in\{4,8,16,32\}.
$$

Because native evaluation with different $$L$$ values begins at different target positions, the final audit recomputes every attention model on the common set

$$
i\ge32.
$$

This removes a small fairness issue without changing the selected model.


| Attention L | Validation accuracy | Validation NLL | Test accuracy | Test NLL | Trainable parameters | Best epoch |
| --- | --- | --- | --- | --- | --- | --- |
| 4 | 0.768909 | 0.598656 | 0.775718 | 0.569040 | 186,576 | 10 |
| 8 | 0.827564 | 0.460849 | 0.826743 | 0.454731 | 186,768 | 9 |
| 16 | 0.836916 | 0.425505 | 0.838392 | 0.415905 | 187,152 | 10 |
| 32 | 0.838228 | 0.424146 | 0.839212 | 0.415362 | 187,920 | 10 |

Validation selects

$$
L=32.
$$

On the same test target positions:

$$
\text{Markov }m=3
=
0.758737,
$$

while

$$
\text{Attention }L=32
=
0.839212.
$$

The difference is approximately

$$
0.080475,
$$

or **8.05 percentage points**.

This clearly establishes that the **selected attention-model configuration** is better for one-step token prediction in this Lorenz experiment.

It does **not** isolate the attention operation as the sole cause of the gain. The attention predictor uses 32 tokens of context and about 188,000 trainable parameters, whereas the baseline is an empirical order-3 model.


# 6. Attention-Weight Interpretation


For selected held-out contexts, the model exposes

$$
A_{ij},
\qquad
j\le i.
$$

The final query row directly determines the attended value mixture used for the next-token prediction.

Two illustrative contexts are examined:

1. a context immediately before a lobe switch;
2. a context remaining within one lobe.

The strongest final-query weights are:


| Example | Rank | Lag | Attention weight |
| --- | --- | --- | --- |
| Near Lobe Switch | 1 | 27 | 0.213204 |
| Near Lobe Switch | 2 | 15 | 0.180897 |
| Near Lobe Switch | 3 | 26 | 0.148691 |
| Near Lobe Switch | 4 | 14 | 0.073291 |
| Within One Lobe | 1 | 8 | 0.289026 |
| Within One Lobe | 2 | 6 | 0.254480 |
| Within One Lobe | 3 | 7 | 0.116261 |
| Within One Lobe | 4 | 25 | 0.091348 |

The selected examples already show that attention is not simply concentrated on the immediately previous token.

The aggregate test-set diagnostic is stronger. Define expected lag

$$
D_i
=
\sum_{j=1}^{L}
A_{Lj}(L-j).
$$

The measured means are:

- non-switch targets:

$$
\bar D_{\mathrm{non-switch}}
=
13.252;
$$

- lobe-switch targets:

$$
\bar D_{\mathrm{switch}}
=
16.260.
$$

With

$$
\Delta t=0.02,
$$

these correspond to approximately 0.265 and 0.325 time units of expected attention lag.

Most importantly, the switch-target mean exceeds the non-switch mean in **all ten held-out test books**, with per-book increases of roughly 2.29 to 3.96 token steps.

This supports an association between impending lobe switches and longer attention reach.

It does **not** prove that attention discovered a causal physical law. The weights can also reflect token identity, discretized geometry, learned positional embeddings, repeated trajectory motifs, or other correlations in the training data.

![Attention matrix near a lobe switch](../result%20images/hw6_1/step6_attention_matrix_near_lobe_switch.png)

![Final-query attention near a lobe switch](../result%20images/hw6_1/step6_attention_profile_near_lobe_switch.png)

![Attention matrix within one lobe](../result%20images/hw6_1/step6_attention_matrix_within_one_lobe.png)

![Final-query attention within one lobe](../result%20images/hw6_1/step6_attention_profile_within_one_lobe.png)


# 7. Autonomous Generation and Lorenz Diagnostics


The generation comparison uses a corrected equal-prefix protocol.

Both the Markov and attention generators receive the same first

$$
32
$$

true held-out tokens.

After that prefix:

- the Markov generator uses only its most recent $$m=3$$ tokens;
- the attention generator uses its most recent $$L=32$$ tokens;
- neither generator receives any future held-out token.

Ten stochastic continuations are generated for each of the ten test books, giving 100 generated sequences per model family.

## 7.1 Coarse attractor-support check

All discrete sequences are mapped to box centers.

A continuous training-state point cloud defines the geometric reference. The threshold is the 95th percentile of validation **box-center** nearest-neighbor distances.

The corrected support results are:


| Dataset | Mean distance | Median distance | p95 distance | Fraction within validation p95 |
| --- | --- | --- | --- | --- |
| True Test | 0.877832 | 0.747042 | 2.220875 | 0.951079 |
| Markov Generated | 0.862722 | 0.723608 | 2.220875 | 0.952022 |
| Attention Generated | 0.890884 | 0.764908 | 2.225257 | 0.947906 |

All three datasets remain near essentially the same coarse tokenized support.

This is only a weak geometric diagnostic. It does not establish the invariant measure or full Lorenz dynamics.

![True, Markov-generated, and attention-generated trajectories](../result%20images/hw6_1/step7_true_markov_attention_xz.png)


# 8. Long-Run Lobe-Switching Behavior


The two coarse lobes are identified using the sign of the token box-center $$x$$ coordinate.

The table reports both the full generated sequence and a second diagnostic after removing the shared 32-token true prefix.


| Dataset | Switches / 1000 (full) | Mean residence time (full) | Switches / 1000 (post-prefix) | Mean residence time (post-prefix) |
| --- | --- | --- | --- | --- |
| True test | 10.960 | 1.778 | 10.837 | 1.795 |
| Markov generated | 10.760 | 1.789 | 10.690 | 1.804 |
| Attention generated | 25.216 | 0.851 | 25.517 | 0.845 |

The main result is striking.

The true trajectories switch at approximately

$$
10.96
$$

times per 1000 sampled steps.

The corrected Markov generator is very close:

$$
10.76.
$$

The attention generator switches much too frequently:

$$
25.22.
$$

Its mean residence time is only about

$$
0.85
$$

time units, compared with approximately

$$
1.78
$$

for both the true trajectories and the Markov generator.

Removing the shared true prefix leaves the same qualitative conclusion.

Therefore:

> **The attention model has higher one-step predictive accuracy but worse autonomous long-run lobe-switching behavior than the Markov model.**

This is an important scientific result rather than a failure of the homework. Supervised next-token training exposes the attention model primarily to true histories. During autonomous generation, prediction errors alter future inputs, creating a distribution shift that can accumulate over time.


# 9. Kirchhoff Invariant Task

Homework 6.1 Task 8 asks for Hamiltonian and Casimir checks when Kirchhoff's equations are selected.

This project uses the dissipative Lorenz system.

Therefore the following Kirchhoff quantities are **not** evaluated or invented for this experiment:

$$
H,
\qquad
C_1=|P|^2,
\qquad
C_2=\Pi\cdot P.
$$

The correct Lorenz-specific diagnostics are the coarse attractor-support and lobe-switching tests above.

# 10. Effect of Attention Context Length

| L | Validation accuracy | Validation NLL | Test accuracy | Test NLL | Parameters | Best epoch |
| --- | --- | --- | --- | --- | --- | --- |
| 4 | 0.768909 | 0.598656 | 0.775718 | 0.569040 | 186,576 | 10 |
| 8 | 0.827564 | 0.460849 | 0.826743 | 0.454731 | 186,768 | 9 |
| 16 | 0.836916 | 0.425505 | 0.838392 | 0.415905 | 187,152 | 10 |
| 32 | 0.838228 | 0.424146 | 0.839212 | 0.415362 | 187,920 | 10 |

The common-target results show:

- a large gain from $$L=4$$ to $$L=8$$;
- a smaller but still meaningful gain from $$L=8$$ to $$L=16$$;
- only a very small additional gain from $$L=16$$ to $$L=32$$.

Thus the supported conclusion is **diminishing returns**, not unlimited improvement from longer memory.

No negative return is observed through $$L=32$$, but longer contexts were not tested.

The selected $$L=32$$ checkpoint reaches its best validation result at the maximum allowed epoch. Therefore this experiment also does not establish that optimization would remain equally easy for substantially longer contexts.

![Attention accuracy versus context length](../result%20images/hw6_1/step9_attention_accuracy_vs_context.png)

![Attention NLL versus context length](../result%20images/hw6_1/step9_attention_nll_vs_context.png)

The figures above were produced using each model's native target range. The common-target table in this section is the authoritative corrected comparison.


# 11. LLM Critic: Prompt and Response

The numerical experiment was fixed before the LLM critique. The complete prompt, including the fixed numerical evidence, was:

```text
You are the critic for a Math 638 experiment on causal attention for tokenized Lorenz trajectories.

The numerical experiment is already complete. Do not redesign the experiment and do not invent new measurements.

You are given:
- the Markov baseline results;
- attention-model validation and test results;
- the selected attention context length;
- attention heatmaps and final-query attention profiles for a within-lobe example and a near-lobe-switch example;
- quantitative expected-attention-lag summaries;
- coarse attractor-support statistics for true, Markov-generated, and attention-generated token trajectories;
- lobe-switching statistics for those trajectories.

Answer the following questions.

1. What do the attention plots appear to focus on?
2. Does the model mostly attend to recent tokens, or does it place substantial weight on earlier parts of the context?
3. Is there evidence that the attention pattern changes near a Lorenz lobe switch compared with ordinary within-lobe motion?
4. Do the numerical attention-lag summaries support the visual impression from the plots?
5. Does the attention model improve next-token prediction over the validation-selected Markov / m-gram baseline?
6. Does longer attention context improve prediction throughout the tested range, or is there a point of diminishing or negative returns?
7. Do the attention weights provide evidence of genuine dynamical structure, or could they simply reflect token-frequency and short-term predictive correlations?
8. Does the attention-generated model reproduce coarse Lorenz behavior better than the Markov generator according to the reported support and lobe-switching diagnostics?
9. Which conclusions are strongly supported by the experiment?
10. Which conclusions would exceed the evidence?

Important constraints:
- The selected system is the dissipative Lorenz system, not the Hamiltonian Kirchhoff system.
- Do not invent conserved Hamiltonian energy, momentum, or Casimir quantities.
- Do not treat a visually attractive attention heatmap as proof that attention has discovered a physical law.
- Do not treat next-token accuracy as proof that the model has recovered the continuous Lorenz vector field.
- Distinguish measured evidence from interpretation and speculation.

Return:
A. Evidence-supported interpretation.
B. Claims that would exceed the evidence.

FIXED NUMERICAL EVIDENCE:

SYSTEM:
Classical Lorenz system with sigma=10, rho=28, beta=8/3.
The system is deterministic, dissipative, and chaotic.

MARKOV TEST RESULTS:
 context_length_m  evaluated_positions  accuracy  exact_history_coverage  unseen_exact_history_fraction  selected_by_validation
                1                12500  0.615440                0.999840                       0.000160                   False
                2                12490  0.698479                0.997998                       0.002002                   False
                3                12480  0.758173                0.994712                       0.005288                    True

ATTENTION VALIDATION RESULTS:
 context_length  trainable_parameters  best_epoch  validation_nll  validation_accuracy  training_seconds                                          checkpoint
              4                186576          10        0.598779             0.768885         13.685280 data\hw6_1\attention_models\attention_context_04.pt
              8                186768           9        0.461279             0.827273         12.656661 data\hw6_1\attention_models\attention_context_08.pt
             16                187152          10        0.425476             0.836761         12.931836 data\hw6_1\attention_models\attention_context_16.pt
             32                187920          10        0.424146             0.838228         12.766392 data\hw6_1\attention_models\attention_context_32.pt

ATTENTION TEST RESULTS:
 context_length  test_nll  test_accuracy  evaluated_positions  selected_by_validation
              4  0.570805       0.775140                12470                   False
              8  0.456078       0.826629                12430                   False
             16  0.416444       0.837976                12350                   False
             32  0.415362       0.839212                12190                    True

VALIDATION-SELECTED ATTENTION MODEL:
{
  "selected_attention_context_length": 32,
  "selection_metric": "lowest validation negative log-likelihood",
  "validation_nll": 0.4241463026714873,
  "validation_accuracy": 0.8382280557834291,
  "candidate_context_lengths": [
    4,
    8,
    16,
    32
  ]
}

COMMON-SUPPORT ATTENTION RESULTS:
     split  context_length  common_start_position  evaluated_positions      nll  accuracy
validation               4                     32                12190 0.598656  0.768909
validation               8                     32                12190 0.460849  0.827564
validation              16                     32                12190 0.425505  0.836916
validation              32                     32                12190 0.424146  0.838228
      test               4                     32                12190 0.569040  0.775718
      test               8                     32                12190 0.454731  0.826743
      test              16                     32                12190 0.415905  0.838392
      test              32                     32                12190 0.415362  0.839212

SELECTED MODEL COMPARISON ON COMMON TARGET POSITIONS:
           model  selected_context  common_start_position  test_accuracy  test_nll
 Markov / m-gram                 3                     32       0.758737       NaN
Causal attention                32                     32       0.839212  0.415362

ATTENTION-LAG SUMMARY:
 is_lobe_switch_target  count      mean    median      std
                 False  12057 13.252350 13.173839 3.755743
                  True    133 16.259528 16.581633 2.160958

GENERATED SUPPORT COMPARISON:
            dataset    representation  count  mean_distance  median_distance  p95_distance  fraction_within_validation_p95_threshold
          true_test token box centers  12510       0.877832         0.747042      2.220875                                  0.951079
   markov_generated token box centers 125100       0.862722         0.723608      2.220875                                  0.952022
attention_generated token box centers 125100       0.890884         0.764908      2.225257                                  0.947906

LOBE-SWITCHING SUMMARY (COMMON 32-TOKEN PREFIX):
,switches_per_1000_steps,switches_per_1000_steps,switches_per_1000_steps,mean_residence_time,mean_residence_time,mean_residence_time
,mean,median,std,mean,median,std
dataset,,,,,,
attention_generated,25.215999999999998,24.0,8.381590215791666,0.8505921524792306,0.8070967741935483,0.28074766510467475
markov_generated,10.76,10.4,2.0384188762743336,1.7891794223346316,1.7871428571428574,0.3399305663792171
true_test,10.959999999999999,10.4,2.7207842006794047,1.7777544147029443,1.7871428571428574,0.3731280744761751


LOBE-SWITCHING SUMMARY AFTER REMOVING THE TRUE PREFIX:
,switches_per_1000_steps,switches_per_1000_steps,switches_per_1000_steps,mean_residence_time,mean_residence_time,mean_residence_time
,mean,median,std,mean,median,std
dataset,,,,,,
attention_generated,25.517241379310345,24.220032840722496,8.63119181534764,0.8446150530926944,0.7995591397849462,0.290714464527174
markov_generated,10.689655172413794,10.673234811165846,2.132604603851557,1.8035958111530712,1.7414285714285713,0.36250358651907555
true_test,10.83743842364532,10.262725779967159,2.731247936480493,1.7952091770974126,1.8084065934065934,0.3784852127429168
```

## Verbatim LLM response

```text
A. Evidence-supported interpretation

1. **What the attention plots focus on.** The final-query attention is sparse and multimodal rather than concentrated only on the immediately previous token. In the selected near-switch example, the largest weights occur at lags 27 (about 0.213), 15 (about 0.181), and 26 (about 0.149). In the selected within-lobe example, the strongest weights occur at lags 8 (about 0.289), 6 (about 0.254), and 7 (about 0.116), with a secondary weight near lag 25 (about 0.091). Thus the model uses nonlocal positions within the 32-token context.

2. **Recent versus earlier context.** The model does not attend only to the recent past. Across the test set, the mean expected attention lag is about 13.25 steps for non-switch targets and 16.26 steps for lobe-switch targets. With dt=0.02, these correspond to about 0.265 and 0.325 time units. This is evidence that appreciable attention mass is often placed well back in the available context.

3. **Change near lobe switches.** There is a reproducible association between switch targets and longer attention lag. The average lag increases by about 3.01 token steps near switches, and the switch-target mean exceeds the non-switch mean in each of the 10 test books. The two plotted examples are consistent with that aggregate result. This supports the statement that the learned attention pattern changes near lobe transitions, but it does not identify a physical mechanism by itself.

4. **Do the quantitative summaries support the plots?** Yes. The plots suggest different temporal emphasis near a switch, and the all-target lag summary supports that observation. However, attention weights are not causal explanations; token identity, position embeddings, recurrent geometric patterns in the attractor, and training correlations can all influence the weights.

5. **Attention versus the Markov baseline.** On the same held-out target positions (all targets at position 32 or later), the validation-selected m=3 Markov model has test accuracy 0.758737, while the selected L=32 attention model has test accuracy 0.839212. The improvement is about 0.0805, or 8.05 percentage points. Therefore the fitted attention predictor is clearly better for one-step token prediction in this experiment. This comparison does not isolate 'attention alone,' because the attention model also has a longer context and many trainable parameters.

6. **Effect of longer attention context.** Longer context helps strongly from L=4 to L=8 and still helps from L=8 to L=16. On common validation targets, accuracy changes from 0.768909 to 0.827564 to 0.836916 to 0.838228 for L=4,8,16,32, while NLL changes from 0.598656 to 0.460849 to 0.425505 to 0.424146. Thus there are clear diminishing returns by L=16 to 32. No negative return is observed within the tested range, but contexts longer than 32 were not tested. Also, the selected L=32 checkpoint occurs at the maximum training epoch, so the experiment does not establish that optimization behavior for still longer contexts would remain easy.

7. **Do attention weights prove genuine dynamics?** No. They provide evidence of a learned temporal association related to the tokenized dynamics, especially because switch targets have systematically longer expected lag. But the experiment cannot distinguish a physically meaningful dynamical feature from correlations caused by token frequency, discretized geometry, or repeated local trajectory motifs. Attention weights should therefore be treated as a diagnostic of model dependence, not as a discovered physical law.

8. **Long-run generation.** The attention generator is not better than the Markov generator on the reported coarse Lorenz diagnostics. All three datasets have roughly 95% of box-center states within the validation-derived support threshold: true test 0.9511, Markov 0.9520, and attention 0.9479. That support test is weak and does not measure the invariant distribution. More importantly, true test trajectories switch at about 10.96 times per 1000 steps, the corrected Markov generator at about 10.76, and the attention generator at about 25.22. Mean lobe residence time is about 1.78 for truth, 1.79 for Markov, and only 0.85 for attention. Removing the common 32-token true prefix leaves the same conclusion. Thus the attention model has better one-step accuracy but substantially worse long-run switching statistics.

9. **Strongly supported conclusions.** (i) causal attention improves held-out one-step token prediction for this Lorenz dataset and discretization; (ii) useful gains from context length diminish after about L=16 in the tested range; (iii) the model frequently assigns weight to nonrecent positions; (iv) lobe-switch targets are associated with longer expected attention lag; and (v) superior one-step accuracy does not imply superior autonomous long-run dynamics.

10. **Claims that would exceed the evidence.** The experiment does not show that the model recovered the Lorenz differential equations, identified the continuous vector field, discovered a conserved quantity, learned a causal physical mechanism, or reproduced the true Lorenz invariant measure or Lyapunov behavior. It also does not establish that attention would outperform the baseline for other dynamical systems, other discretizations, or other training-set sizes, because those factors were not varied.

B. Claims that would exceed the evidence

The strongest overclaim would be to interpret visually attractive attention patterns as proof that the network discovered Lorenz physics. The experiment instead supports a narrower statement: attention learns useful nonlocal predictive correlations in the tokenized trajectories, and those correlations change systematically near lobe switches. The long-run generation failure shows why prediction accuracy and physical fidelity must be evaluated separately.
```


# 12. Human Critique of the LLM Response

The LLM response is mostly careful and is consistent with the fixed numerical evidence. It correctly avoids treating the Lorenz system as Hamiltonian, does not invent energy or Casimir invariants, and distinguishes one-step prediction from autonomous long-run generation. Its most useful point is the contrast between the attention model's higher next-token accuracy and its much worse lobe-switching statistics. That contrast directly shows why a sequence model can be locally predictive without reproducing the long-run dynamics.

I agree with the LLM that the attention pattern is not purely local. The selected profiles place substantial mass at nonrecent lags, and the all-target statistic is stronger evidence than either individual heatmap: the mean expected lag is about 13.25 for ordinary targets and 16.26 for switch targets. The difference is also positive in every held-out test book. This supports an association between lobe switching and longer attention reach.

However, I would weaken any wording that calls this a discovered dynamical mechanism. Attention weights are model-internal dependence weights, not causal attribution. The model may be exploiting repeated token patterns, the geometry created by the box discretization, position embeddings, or token-frequency correlations. The two hand-selected plots are illustrative rather than independent evidence; the aggregate lag calculation is the appropriate basis for the claim.

I would also qualify the statement that 'attention improves over Markov.' On common target positions the numerical result is clear: 83.921% versus 75.874% test accuracy. But this comparison does not isolate the attention operation alone. The selected attention model has a 32-token context and roughly 188k trainable parameters, while the baseline is an m=3 empirical model. Therefore the supported conclusion is that **this attention-based predictor configuration** performs better, not that attention is solely responsible for the entire gain.

The attractor-support diagnostic also needs cautious interpretation. Roughly 95% of true, Markov-generated, and attention-generated box centers lie within the validation-derived threshold, so the attention generator does not obviously leave the coarse training support. But this is a weak geometric check and cannot establish the correct invariant measure or chaotic dynamics. The lobe-switching statistic is much more discriminating here: the attention generator switches about 25.22 times per 1000 steps versus 10.96 for truth, whereas the corrected Markov generator is about 10.76.

Finally, the context-length experiment supports diminishing returns, not an unlimited benefit from longer memory. The gain from L=16 to L=32 is very small compared with the gain from L=4 to L=8. Because only one Lorenz discretization and one training-set size were tested, I would not generalize the conclusion to other systems, box resolutions, or data regimes.

**Final critique:** the LLM interpretation is useful when it stays close to the measured accuracy, lag, support, and switching statistics. The claims become too strong if the attention heatmaps are treated as physical explanations. The safest conclusion is that attention learns nonlocal predictive correlations that are measurably different near lobe switches, improves one-step prediction in this experiment, but does not by itself recover the Lorenz dynamics and actually performs worse than the Markov generator on long-run lobe-switching behavior.

# 13. Reproducibility, Audit Corrections, and Limitations

## 13.1 Audit corrections incorporated in the final interpretation

Three corrections or qualifications are treated as authoritative in this report.

### Common-target model comparison

Different attention context lengths originally began evaluation at different target positions. The corrected comparison evaluates every attention model on positions

$$
i\ge32.
$$

The model choice and main conclusion are unchanged.

### Equal-prefix generation

The original generation comparison gave the Markov model only 3 true seed tokens but the attention model 32.

The corrected comparison gives both generators the same 32-token true held-out prefix and then runs both autonomously.

### Completed LLM critic

The Step 10 response and separate human critique are now complete.

## 13.2 Independent verification

The corrected headline numbers were independently recomputed from the saved checkpoints and generated sequences:

- attention common-support accuracy at $$L=32$$:

$$
0.839212;
$$

- Markov common-support accuracy at $$m=3$$:

$$
0.758737;
$$

- true / Markov / attention switching rates:

$$
10.96,\quad10.76,\quad25.216
$$

per 1000 steps;

- all saved Markov and attention generated sequences exactly match their corresponding true held-out books for the first 32 tokens.

## 13.3 Remaining low-severity hygiene issue

The preprocessing notebook materializes aggregate validation/test token-frequency columns before fitting.

Those descriptive columns are not used by the Markov counts, neural gradient updates, early stopping, or context selection, so they do not contaminate the reported fitted-model results.

A stricter blind-test workflow would postpone those test summaries until final evaluation.

## 13.4 Reproducibility note

The common-support audit CSV files are included in `data/hw6_1/` and contain correct recomputed values. In the updated attention-training notebook, the final audit cell **reads** these corrected CSVs rather than recomputing them from the saved checkpoints inside that notebook.

This does not invalidate the results, but a maximally self-contained version would include the recomputation code directly.

## 13.5 Scientific limitations

The experiment does not establish that:

- the model recovered the continuous Lorenz vector field;
- the attention weights are causal physical explanations;
- the generated invariant measure is correct;
- the Lyapunov spectrum or entropy is reproduced;
- attention will outperform Markov models for other dynamical systems;
- the result will persist for other discretizations or training-set sizes.

These remain outside the evidence produced by this experiment.

# Overall Conclusions

Homework 6.1 demonstrates the distinction between **predictive memory** and **dynamical fidelity**.

The causal-attention model clearly improves held-out one-step token prediction relative to the selected Markov baseline:

$$
83.921\%
\quad\text{versus}\quad
75.874\%.
$$

The learned attention is also genuinely nonlocal within the tested 32-token window. Lobe-switch targets exhibit a systematically larger expected attention lag than ordinary targets.

However, those advantages do not translate into better autonomous long-run Lorenz behavior.

The attention generator switches between lobes more than twice as often as the true trajectories, while the simpler Markov generator remains much closer to the true switching statistics.

Therefore the main scientific conclusion is:

> **The selected causal-attention model learns useful nonlocal temporal correlations and improves one-step prediction, but improved local prediction does not by itself imply recovery of the long-run dynamics of the original system.**

This distinction is central to interpreting learned sequence models for dynamical systems.